# 11 - Final Exports

Following source `11_morphotope_geometry.ipynb` + `ext_15_data_products.ipynb`:
1. Dissolve tessellation cells into morphotope polygons
2. Attach cluster hierarchy labels to every building
3. Export GeoJSON (WGS84) for the TypeScript viewer app
4. Export GPKG, CSV, and taxonomy JSON

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else ".")

import numpy as np
import pandas as pd
import geopandas as gpd
import json
from libpysal.graph import Graph
from pathlib import Path

OUT = Path("outputs/11_exports")
OUT.mkdir(parents=True, exist_ok=True)

## Load data

In [2]:
tessellation = gpd.read_parquet("outputs/02_tessellation/tessellation.parquet")
buildings    = gpd.read_file("outputs/01_preprocessed/buildings.gpkg")
clusters     = pd.read_parquet("outputs/09_hierarchy/tessellation_clusters_final.parquet")
morph_labels = pd.read_parquet("outputs/06_morphotopes/tessellation_labels_postprocessed.parquet")["morphotope_label"]

print(f"Tessellation: {len(tessellation):,}")
print(f"Buildings: {len(buildings):,}")
print(f"Clusters: {clusters.shape}")

Tessellation: 274,046
Buildings: 269,107
Clusters: (269107, 10)


## 1. Dissolve tessellation into morphotope polygons

Source `11_morphotope_geometry.ipynb`: dissolve non-noise by morphotope label,
dissolve noise by contiguity components, buffer(0.01) to close gaps.

In [3]:
# Assign morph labels + clusters to tessellation (only building cells, index >= 0)
tessellation["morph"] = morph_labels.reindex(tessellation.index)
tessellation = tessellation.join(clusters, how="left")

# Drop empty enclosure cells (negative index, no building)
tess_bldg = tessellation[tessellation["morph"].notna()].copy()

# Non-noise morphotopes: dissolve by morphotope label
is_noise = tess_bldg["morph"].str.endswith('_-1')
non_noise = tess_bldg[~is_noise].copy()
morph_geom = non_noise[["morph", "geometry"]].dissolve(by="morph").reset_index()

# Noise: dissolve by contiguity components
noise = tess_bldg[is_noise].copy()
if len(noise) > 0:
    noise_graph = Graph.build_fuzzy_contiguity(noise, buffer=0.1)
    noise["noise_comp"] = noise_graph.component_labels.values
    noise_geom = noise[["noise_comp", "geometry"]].dissolve(by="noise_comp").reset_index()
    noise_geom["morph"] = "noise_" + noise_geom["noise_comp"].astype(str)
    noise_geom = noise_geom[["morph", "geometry"]]
    morph_geom = pd.concat([morph_geom, noise_geom], ignore_index=True)

# Close tiny gaps
morph_geom["geometry"] = morph_geom.geometry.buffer(0.01)

# Attach cluster info (modal cluster per morphotope)
morph_cluster = tess_bldg.groupby("morph").agg(
    final=("final", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan),
    building_count=("morph", "count"),
)
for col in [c for c in clusters.columns if c.startswith('level_')]:
    morph_cluster[col] = tess_bldg.groupby("morph")[col].agg(
        lambda x: x.mode().iloc[0] if len(x.dropna()) > 0 and len(x.mode()) > 0 else np.nan
    )

morph_geom = morph_geom.merge(morph_cluster, left_on="morph", right_index=True, how="left")
print(f"Morphotope polygons: {len(morph_geom):,}")

Morphotope polygons: 2,477


## 2. Building-level export

Attach morphotope ID and hierarchy labels to every building.

In [4]:
# Buildings align 1:1 with tessellation cells (building index = tess positive index)
bldg_export = buildings.copy()
bldg_export["morphotope_id"] = morph_labels.reindex(bldg_export.index)

for col in clusters.columns:
    bldg_export[col] = clusters[col].reindex(bldg_export.index)

print(f"Building export: {len(bldg_export):,} rows, {len(bldg_export.columns)} columns")

Building export: 269,107 rows, 29 columns


## 3. Reproject to WGS84 and export GeoJSON

In [5]:
# Morphotopes GeoJSON
morph_wgs = morph_geom.to_crs("EPSG:4326")
# Round floats
for c in morph_wgs.select_dtypes(include='float64').columns:
    morph_wgs[c] = morph_wgs[c].round(4)
morph_wgs.to_file(OUT / "winnipeg_morphotopes.geojson", driver="GeoJSON")
print(f"Morphotopes GeoJSON: {len(morph_wgs):,} features")

# Tessellation cells GeoJSON (slim: keep key columns only)
tess_wgs = tessellation.to_crs("EPSG:4326")
tess_keep = [c for c in ["morph", "final", "initially_noise"] + 
             [c for c in clusters.columns if c.startswith('level_')] + ["geometry"]
             if c in tess_wgs.columns]
tess_slim = tess_wgs[tess_keep].copy()
tess_slim.to_file(OUT / "winnipeg_tessellation.geojson", driver="GeoJSON")
print(f"Tessellation GeoJSON: {len(tess_slim):,} features")

Morphotopes GeoJSON: 2,477 features
Tessellation GeoJSON: 274,046 features


## 4. Export GeoPackage + CSV + taxonomy JSON

In [6]:
# GeoPackage with all layers
gpkg = OUT / "winnipeg_taxonomy.gpkg"
morph_wgs.to_file(gpkg, layer="morphotopes", driver="GPKG")
bldg_export.to_crs("EPSG:4326").to_file(gpkg, layer="buildings", driver="GPKG", mode="a")
print(f"GPKG: {gpkg}")

# CSV summary
summary = morph_geom.drop(columns="geometry")
summary.to_csv(OUT / "summary_stats.csv", index=False)

# Taxonomy type profiles JSON (for TypeScript app)
type_profiles = []
for level_col in sorted([c for c in clusters.columns if c.startswith('level_')]):
    level_data = morph_geom.groupby(level_col).agg(
        count=("morph", "count"),
        buildings=("building_count", "sum"),
    ).reset_index()
    type_profiles.append({
        "level": level_col,
        "clusters": level_data.to_dict(orient="records"),
    })

with open(OUT / "taxonomy_hierarchy.json", "w") as f:
    json.dump(type_profiles, f, indent=2, default=str)

print(f"\nAll exports saved to {OUT}:")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

GPKG: outputs\11_exports\winnipeg_taxonomy.gpkg

All exports saved to outputs\11_exports:
  summary_stats.csv (0.1 MB)
  taxonomy_hierarchy.json (0.0 MB)
  winnipeg_morphotopes.geojson (154.7 MB)
  winnipeg_taxonomy.gpkg (151.6 MB)
  winnipeg_tessellation.geojson (392.3 MB)
